# Project 2
#  A structured data mining analysis on dataset extracted from images of dry beans. The goal is to classify each instance into one of seven bean types.

### By Wasim Raja Mondal

<a id="99"></a>


<b>Program Sections</b>
<ul>
<li><a href = "#0">  Part 1: Data Understanding  </a></li>
<li><a href = "#1"> Part 2: Feature Engineering and Model Performance Comparison</a></li>
<li><a href = "#2"> Part 3: Model Performance comparison between original vs PCA</a></li> 

In [1]:
!pip install numpy #installing numpy
!pip install pandas #installing pandas
!pip install matplotlib #installing matplotlib
!pip install seaborn #installing seaborn
!pip install scikit-learn #installing scikit learn
import zipfile
import io
import numpy as np  #importing numpy
import pandas as pd #importing pandas
import seaborn as sns #importing seaborn
import pandas as pd #importing pandas
import matplotlib.pyplot as plt
pd.set_option('display.max_columns',500) #allows for up to 500 columns to be displayed when viewing a dataframe
plt.style.use("seaborn-v0_8-whitegrid") ##a style that can be used for plots
import datetime as dt #importing datetime


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Part 1: Data Understanding 

In [2]:
# Open ZIP and load Excel file manually
with zipfile.ZipFile('data/dry+bean+dataset.zip') as z:
    with z.open('DryBeanDataset/Dry_Bean_Dataset.xlsx') as file:
        df_drybean = pd.read_excel(file, engine='openpyxl')

In [3]:
df_drybean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             13611 non-null  int64  
 1   Perimeter        13611 non-null  float64
 2   MajorAxisLength  13611 non-null  float64
 3   MinorAxisLength  13611 non-null  float64
 4   AspectRation     13611 non-null  float64
 5   Eccentricity     13611 non-null  float64
 6   ConvexArea       13611 non-null  int64  
 7   EquivDiameter    13611 non-null  float64
 8   Extent           13611 non-null  float64
 9   Solidity         13611 non-null  float64
 10  roundness        13611 non-null  float64
 11  Compactness      13611 non-null  float64
 12  ShapeFactor1     13611 non-null  float64
 13  ShapeFactor2     13611 non-null  float64
 14  ShapeFactor3     13611 non-null  float64
 15  ShapeFactor4     13611 non-null  float64
 16  Class            13611 non-null  object 
dtypes: float64(1

In [4]:
df_drybean.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [5]:
df_drybean['Class'].value_counts()

Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

## Part 2: Feature Engineering and Model Performance Comparison

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, chi2

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# -------------------------------
# Load data
# -------------------------------
# If already loaded, you can skip this line
# df_drybean = pd.read_excel("data/Dry_Bean_Dataset.xlsx")

# -------------------------------
# Encode target
# -------------------------------
le = LabelEncoder()
df_drybean["Class_encoded"] = le.fit_transform(df_drybean["Class"])

X = df_drybean.drop(["Class", "Class_encoded"], axis=1)
y = df_drybean["Class_encoded"]

print("Class mapping:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

# -------------------------------
# Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------------
# Original scaled data
# -------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------
# Feature selection: Variance Threshold
# -------------------------------
var_selector = VarianceThreshold(threshold=0.01)

X_train_var = var_selector.fit_transform(X_train)
X_test_var = var_selector.transform(X_test)

selected_features_var = X.columns[var_selector.get_support()]

print("\nVariance Threshold")
print("Original:", X_train.shape[1], "Selected:", X_train_var.shape[1])
print("Selected features:", list(selected_features_var))

# Scale variance-selected features
scaler_var = StandardScaler()
X_train_var_scaled = scaler_var.fit_transform(X_train_var)
X_test_var_scaled = scaler_var.transform(X_test_var)

# -------------------------------
# Feature selection: Mutual Information
# -------------------------------
mi_selector = SelectKBest(score_func=mutual_info_classif, k=10)

X_train_mi = mi_selector.fit_transform(X_train, y_train)
X_test_mi = mi_selector.transform(X_test)

selected_features_mi = X.columns[mi_selector.get_support()]

print("\nMutual Information")
print("Selected:", X_train_mi.shape[1], "features")
print("Selected features:", list(selected_features_mi))

# Scale MI-selected features
scaler_mi = StandardScaler()
X_train_mi_scaled = scaler_mi.fit_transform(X_train_mi)
X_test_mi_scaled = scaler_mi.transform(X_test_mi)

# -------------------------------
# Feature selection: Chi-square
# -------------------------------
# Chi-square requires nonnegative features
scaler_chi2 = MinMaxScaler()

X_train_chi2_scaled_all = scaler_chi2.fit_transform(X_train)
X_test_chi2_scaled_all = scaler_chi2.transform(X_test)

chi2_selector = SelectKBest(score_func=chi2, k=10)

X_train_chi2 = chi2_selector.fit_transform(X_train_chi2_scaled_all, y_train)
X_test_chi2 = chi2_selector.transform(X_test_chi2_scaled_all)

selected_features_chi2 = X.columns[chi2_selector.get_support()]

print("\nChi-square")
print("Selected:", X_train_chi2.shape[1], "features")
print("Selected features:", list(selected_features_chi2))

# Already MinMax-scaled, but StandardScaler can still be used for models like SVM/KNN/MLP
scaler_chi2_final = StandardScaler()
X_train_chi2_scaled = scaler_chi2_final.fit_transform(X_train_chi2)
X_test_chi2_scaled = scaler_chi2_final.transform(X_test_chi2)

# ================================
# Model evaluation function
# ================================
def evaluate_models(Xtr, Xte, ytr, yte, feature_set_name):
    
    models = {
        "Decision Tree": DecisionTreeClassifier(
            random_state=42
        ),

        "KNN": KNeighborsClassifier(
            n_neighbors=5
        ),

        "AdaBoost": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
            n_estimators=100,
            random_state=42
        ),

        "SVM": SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ),

        "MLP": MLPClassifier(
            hidden_layer_sizes=(128, 64),
            max_iter=500,
            random_state=42
        )
    }
    
    results = []
    confusion_matrices = {}
    
    for model_name, model in models.items():
        
        print("\n" + "="*80)
        print(f"Feature Set: {feature_set_name}")
        print(f"Model: {model_name}")
        print("="*80)
        
        model.fit(Xtr, ytr)
        y_pred = model.predict(Xte)
        
        acc = accuracy_score(yte, y_pred)
        f1_macro = f1_score(yte, y_pred, average="macro")
        f1_weighted = f1_score(yte, y_pred, average="weighted")
        cm = confusion_matrix(yte, y_pred)
        
        print("Accuracy:", acc)
        print("Macro F1:", f1_macro)
        print("Weighted F1:", f1_weighted)
        
        print("\nClassification Report:")
        print(classification_report(yte, y_pred, target_names=le.classes_))
        
        print("\nConfusion Matrix:")
        print(cm)
        
        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "Accuracy": acc,
            "Macro F1": f1_macro,
            "Weighted F1": f1_weighted
        })
        
        confusion_matrices[(feature_set_name, model_name)] = cm
    
    return pd.DataFrame(results), confusion_matrices

# ================================
# Run experiments
# ================================

results_original, cm_original = evaluate_models(
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    "Original Scaled"
)

results_variance, cm_variance = evaluate_models(
    X_train_var_scaled,
    X_test_var_scaled,
    y_train,
    y_test,
    "Variance Threshold"
)

results_mi, cm_mi = evaluate_models(
    X_train_mi_scaled,
    X_test_mi_scaled,
    y_train,
    y_test,
    "Mutual Information"
)

results_chi2, cm_chi2 = evaluate_models(
    X_train_chi2_scaled,
    X_test_chi2_scaled,
    y_train,
    y_test,
    "Chi-square"
)

# -------------------------------
# Combine all results
# -------------------------------
all_results = pd.concat(
    [
        results_original,
        results_variance,
        results_mi,
        results_chi2
    ],
    ignore_index=True
)

all_results = all_results.sort_values(
    by="Weighted F1",
    ascending=False
)

print("\n\nFinal Model Comparison:")
print(all_results)

Class mapping:
{'BARBUNYA': np.int64(0), 'BOMBAY': np.int64(1), 'CALI': np.int64(2), 'DERMASON': np.int64(3), 'HOROZ': np.int64(4), 'SEKER': np.int64(5), 'SIRA': np.int64(6)}

Variance Threshold
Original: 16 Selected: 7
Selected features: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'ConvexArea', 'EquivDiameter']

Mutual Information
Selected: 10 features
Selected features: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'ConvexArea', 'EquivDiameter', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3']

Chi-square
Selected: 10 features
Selected features: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'ConvexArea', 'EquivDiameter', 'Compactness', 'ShapeFactor2', 'ShapeFactor3']

Feature Set: Original Scaled
Model: Decision Tree
Accuracy: 0.8920308483290489
Macro F1: 0.9080614184713196
Weighted F1: 0.891630337288096

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA 

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

Accuracy: 0.9056188027910393
Macro F1: 0.9193613211520154
Weighted F1: 0.9060176737374798

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.98      0.82      0.89       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.89      0.95      0.92       326
    DERMASON       0.90      0.90      0.90       709
       HOROZ       0.97      0.93      0.95       386
       SEKER       0.93      0.93      0.93       406
        SIRA       0.81      0.87      0.84       527

    accuracy                           0.91      2723
   macro avg       0.93      0.92      0.92      2723
weighted avg       0.91      0.91      0.91      2723


Confusion Matrix:
[[218   0  32   0   1   4  10]
 [  0 104   0   0   0   0   0]
 [  4   0 311   0   5   2   4]
 [  0   0   0 637   0  14  58]
 [  1   0   6   7 358   0  14]
 [  0   0   0   8   0 379  19]
 [  0   0   1  53   5   9 459]]

Feature Set: Chi-square
Model: MLP
Accuracy: 0.9155

## Part 3: Original vs PCA comparison

In [7]:
# ================================
# PCA Experiment: Original vs PCA
# Dry Bean Multiclass Classification
# ================================

from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# -------------------------------
# Define models
# -------------------------------
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=100,
        random_state=42
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=500,
        random_state=42
    )
}

# -------------------------------
# PCA setup
# -------------------------------
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original features:", X_train_scaled.shape[1])
print("PCA components:", X_train_pca.shape[1])
print("Explained variance ratio:", pca.explained_variance_ratio_.sum())

# -------------------------------
# Function to evaluate models
# -------------------------------
def evaluate_pca_models(Xtr, Xte, ytr, yte, feature_set_name):
    
    results = []
    
    for model_name, model in models.items():
        
        print("\n" + "="*80)
        print(f"Feature Set: {feature_set_name}")
        print(f"Model: {model_name}")
        print("="*80)
        
        model.fit(Xtr, ytr)
        y_pred = model.predict(Xte)
        
        acc = accuracy_score(yte, y_pred)
        f1_macro = f1_score(yte, y_pred, average="macro")
        f1_weighted = f1_score(yte, y_pred, average="weighted")
        cm = confusion_matrix(yte, y_pred)
        
        print("Accuracy:", acc)
        print("Macro F1:", f1_macro)
        print("Weighted F1:", f1_weighted)
        
        print("\nClassification Report:")
        print(classification_report(yte, y_pred, target_names=le.classes_))
        
        print("\nConfusion Matrix:")
        print(cm)
        
        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "Accuracy": acc,
            "Macro F1": f1_macro,
            "Weighted F1": f1_weighted
        })
    
    return pd.DataFrame(results)

# -------------------------------
# Evaluate original scaled data
# -------------------------------
results_original_pca_compare = evaluate_pca_models(
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    "Original Scaled"
)

# -------------------------------
# Evaluate PCA data
# -------------------------------
results_pca = evaluate_pca_models(
    X_train_pca,
    X_test_pca,
    y_train,
    y_test,
    "PCA 95% Variance"
)

# -------------------------------
# Compare results
# -------------------------------
pca_comparison = pd.concat(
    [results_original_pca_compare, results_pca],
    ignore_index=True
)

pca_comparison = pca_comparison.sort_values(
    by="Weighted F1",
    ascending=False
)

print("\n\nFinal PCA Comparison:")
print(pca_comparison)

# Optional: pivot table for easy comparison
comparison_table = pca_comparison.pivot(
    index="Model",
    columns="Feature Set",
    values=["Accuracy", "Macro F1", "Weighted F1"]
)

print("\n\nOriginal vs PCA Comparison Table:")
print(comparison_table)

Original features: 16
PCA components: 4
Explained variance ratio: 0.9505407948838052

Feature Set: Original Scaled
Model: Decision Tree
Accuracy: 0.8920308483290489
Macro F1: 0.9080614184713196
Weighted F1: 0.891630337288096

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.88      0.91      0.89       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.92      0.91      0.92       326
    DERMASON       0.88      0.90      0.89       709
       HOROZ       0.94      0.91      0.93       386
       SEKER       0.91      0.95      0.93       406
        SIRA       0.83      0.79      0.81       527

    accuracy                           0.89      2723
   macro avg       0.91      0.91      0.91      2723
weighted avg       0.89      0.89      0.89      2723


Confusion Matrix:
[[240   0  16   1   0   2   6]
 [  0 104   0   0   0   0   0]
 [ 16   0 298   0   6   3   3]
 [  0   0   0 635   2  15  57]
 [  8   0  

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

Accuracy: 0.9001101726037458
Macro F1: 0.9011212604063269
Weighted F1: 0.8991787764182645

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.87      0.69      0.77       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.80      0.89      0.85       326
    DERMASON       0.92      0.92      0.92       709
       HOROZ       0.96      0.96      0.96       386
       SEKER       0.94      0.95      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.90      2723
   macro avg       0.91      0.90      0.90      2723
weighted avg       0.90      0.90      0.90      2723


Confusion Matrix:
[[182   0  66   0   1   4  12]
 [  0 104   0   0   0   0   0]
 [ 25   0 291   0   6   2   2]
 [  0   0   0 654   0  11  44]
 [  0   0   5   6 370   0   5]
 [  1   0   0   8   0 384  13]
 [  2   0   0  44   8   7 466]]

Feature Set: PCA 95% Variance
Model: MLP
Accuracy: 